# 2022_12_07

이전 회의 때 

* ewma를 2번 적용한 data는 예측을 그나마 잘 하는 모습이고 현재 ewma를 한번 적용한 그래프는 예측을 잘 하지 못함 

ewma를 2번 적용을 하면 사건 발생 이전에 값이 있고 이를 통해서 다음에 사건이 발생할 것이다라는 예측을 하는데  
ewma를 1번 적용하면 이러한 전조증상이 없어서 예측을 하지 못함(이때의 그래프도 실제 사간 발생 이후에 예측을 함)  

* 기존의 graph기반으 접근법으로 하면 사건이 발생 이전의 근처의 동에서 값을 가져오므로 사건 발생 이전의 값을 가져와서 ewma와 2번 적용한 것과 유사한 효과를 낼 수 있을거 같음  
(도표 추가)  

##  graph 

기존의 graph 기반의 target값을 생성하는 방법을 이용해서 학습을 진행  
* nei0 , nei1 , nei2의 산술 가중 평균을 이용해서 생성
* 생선된 데이터와  ewma를 두번 적용한 데이터와 비교

#### graph result 
| nei0 | nei1 | nei1 | result |
|---|---|:---:|---|
| 1 | 1 | 0 | http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_06_TEST_best_graph_hprams/result_graph__1_1_0.ipynb |
| 1 | 0.5 | 0 | http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_06_TEST_best_graph_hprams/result_graph__1_0.5_0.ipynb |
| 2 | 0.5 | 0.1 | http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_06_TEST_best_graph_hprams/result_graph__2_0.5_0.1.ipynb |



## visualization

1. Quantile별 prediction의 plot
2. ewma2 와 prediction의 비교


## ewma test  
현재 기존의 ewma를 2번 적용한 방식에서 한 번만 사용했다. ewma와 , ewma를 뒤집은 data를 둘의 평균을 사용했는데 지금 뒤집은 data를 이용해서 test하고 있음 이는 실제 data와 비교해도 맞지 않고 그래프 아래의 면적이 좁게 나와서 학습하기에 좋지 않은 방식이다.  

```python
def moving_average_alpha_forward(df: pd.DataFrame, unit: float):
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    # forward df
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        inv_dong_df = df[df['h_dong'] == dong][::-1]
        dong_df['count'] = dong_df['count'].ewm(alpha = unit).mean()
        for_ewma = dong_df['count'].ewm(alpha = unit).mean()
        back_ewma = inv_dong_df['count'].ewm(alpha = unit).mean()

        df['count'][dong_df.index] = back_ewma 
    df['count'] = df['count']  /  df['count'].max() * (max_value)    
    return df

def moving_average_alpha_backward(df: pd.DataFrame, unit: float):
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        inv_dong_df = df[df['h_dong'] == dong][::-1]
        dong_df['count'] = dong_df['count'].ewm(alpha = unit).mean()
        for_ewma = dong_df['count'].ewm(alpha = unit).mean()
        back_ewma = inv_dong_df['count'].ewm(alpha = unit).mean()
        df['count'][dong_df.index] = for_ewma 
    df['count'] = df['count']  /  df['count'].max() * (max_value)
    
    return df
```

함수들 실행 결과  

![](https://i.imgur.com/ac91ONd.png)
![](https://i.imgur.com/soyV466.png)

코드를 다시 확인해보니 ewma는 2번씩 적용하고 있었다.  
이를 수정함  

```python
def moving_average_alpha(df: pd.DataFrame, unit: float):
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    # forward df
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        max_value = dong_df['count'].max()
        back_ewma = dong_df['count'].ewm(alpha = unit).mean()
        back_ewma = back_ewma / back_ewma.max() * (max_value)
        #print(f'{dong} max : {max_value}   ewma_max :{back_ewma.max()}')
        df['count'][dong_df.index] = back_ewma 
    
    #print(max_value , df['count'].max())
    return df
```

코드 변경 결과 

![](https://i.imgur.com/dwxcdlR.png)
![](https://i.imgur.com/4Cz7WK7.png)


* 현재 plot을 보면 유사한 패턴이라도 scale이 달라서 학습이 이상하게 될거 같음 -> 다른 scaling을 적용해야 함?
* 사건이 근처에서 발생하면 ewma 때문에 값이 더 올라가고 이 값을 가지고 scaling을 해서 값이 떨어지는 경우가 나타남  
* 현재 linear scaling으로 값을 올려주었는데 이를 다른 exp scailng과 같은 다른 방법을 적용해야 될까? 

### emwa의 한계 


* 전체 : 실제의 값보다 낮은 값이 나오고 있음 이는 사건 발생이 겹치면 겹칠수록 이런 증상이 심해짐 

![](https://i.imgur.com/RZbzEn7.png)
* 강남동 : 1월 17일과 1월 27의 경우에는 둘이 같이 사건이 한 번 발생했는데  ewma를 적용하면 다른 값으로 나옴

![](https://i.imgur.com/Lp9tD4X.png)
* 근화동 : 붙어 있으면 같은 사건 발생이라도 다른 패턴이 나타남 

![](https://i.imgur.com/6XYidHs.png)

* 조운동 : ewma의 한계점이 명확히 나타남 

![](https://i.imgur.com/aJmWb4U.png)
* 효자1동 : 1월 26일 ~ 2월 1일까지 너무 붙어 있어서 같은 사건발생인데 값이 감소하고 있음 

## ewma_fix

ewma 코드를 고치고 난 이후의 원본값과 ewma 값을 비교  

> result_notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_06_TEST_ewma_fix/ewma_test_plot.ipynb

## meeting
ewma의 scaling을 하는 것은 문제가 없어보임 -> 이런 패턴들로 인해서 학습이 더 잘됨  
ewma factor와 cut off를 잘 조합해서 좋은 성능을 보이는 hparmas 찾기   
peak_v2방식을 이용해서 좋은 성능을 보이는 ewma를 찾기  